In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"  # Limit OpenMP
os.environ["MKL_NUM_THREADS"] = "5"  # Limit MKL (Intel Math Kernel Library)
os.environ["OPENBLAS_NUM_THREADS"] = "5"  # Limit OpenBLAS
os.environ["NUMEXPR_MAX_THREADS"] = "5"  # Limit NumExpr if installed

In [ ]:
import itertools
from pathlib import Path
import re
from typing import Literal

from matplotlib.colors import CenteredNorm
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pickle
from scipy.stats import ttest_ind, pearsonr, spearmanr
import seaborn as sns
import textgrid
import torch
from tqdm.auto import tqdm
# from tqdm import tqdm

from src.data import get_electrode_df, add_metadata_features
from src.models.causal4 import run_causal4_analysis

In [ ]:
epochs_path = "outputs/epochs_preprocessed"
tg_dir = "textgrids"

In [ ]:
all_epoch_paths = list(Path(epochs_path).glob("*.fif"))

In [ ]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epo", str(path))[0]
    epochs[subject_name] = mne.read_epochs(str(path))
    epochs[subject_name].metadata = add_metadata_features(epochs[subject_name].metadata)

In [ ]:
electrode_df = pd.concat([get_electrode_df(subject_name) for subject_name in epochs.keys()],
                         keys=epochs.keys(), names=["subject"])
electrode_df["roi"] = electrode_df.roi.astype(str)
electrode_df = electrode_df.droplevel("electrode_name")

# Drop electrodes metadata which don't have corresponding data
for subject, epochs_i in epochs.items():
    electrode_df.loc[subject, "keep"] = np.arange(len(electrode_df.loc[subject])) < len(epochs_i.info["ch_names"])
electrode_df = electrode_df[electrode_df["keep"]].drop(columns="keep")

electrode_df

## Model sketch

In [ ]:
import json
with open("causal_candidate_populations.json", "r") as f:
    populations = json.load(f)

In [ ]:
populations

In [ ]:
from tqdm.auto import trange

all_results = []
all_results_meta = {}
populations_dict = {}

# NB, with split_strategy="extreme", what's being randomized across runs
# is only the shuffling in the nested CV splits for estimating phase 1/2
# encoders.
n_repeats = 10
split_strategy = "extreme" # "stratify" or "extreme"

for repeat in trange(n_repeats):
    for population in tqdm(populations):
        result = run_causal4_analysis(
            epochs, subject=population["subject"],
            phoneme_pair=population["phoneme_pair"],
            population_A=population["electrodes_a"],
            population_A_window=population["window_a"],
            population_B=population["electrodes_b"],
            population_B_window=population["window_b"],
            split_strategy=split_strategy,
        )

        name = f"{population['subject']}_{population['phoneme_pair']}_A{population['cluster_a']}_B{population['cluster_b']}"
        key = (name, repeat)
        populations_dict[key] = population
        all_results_meta[key] = result

        correlation_p = np.corrcoef(
            result.p_gt_phoneme,
            result.p_gt_lexical_evidence,
        )[0, 1]
        
        row = pd.Series({
            "name": name,
            "repeat": repeat,
            "corr": correlation_p,
            "window_a_start": population["window_a"][0],
            "window_a_end": population["window_a"][1],
            "window_b_start": population["window_b"][0],
            "window_b_end": population["window_b"][1],

            "phase1_scores": result.phase1_test_scores.mean(),
            "phase1_val_score": result.phase1_val_score,
            "phase2_scores": result.phase2_test_scores.mean(),
        })

        all_results.append(row)
        
        

In [ ]:
all_results_df = pd.DataFrame(all_results)
all_results_df = all_results_df.set_index(["name", "repeat"])

In [ ]:
all_results_df.groupby("name").agg(["mean", "std"]).sort_values(("corr", "mean"))

In [ ]:
order = all_results_df.groupby("name").agg(["mean", "std"]).sort_values(("corr", "mean")).index
g = sns.catplot(data=all_results_df.reset_index(), x="corr", y="name", order=order, kind="box", aspect=0.5, height=7)
g.axes.flat[0].axvline(0, color="k", linestyle="--")
None

In [ ]:
sns.displot(all_results_df["corr"])

In [ ]:
plot_key = "EC282_pb_A3_B4"
# merge estimated probabilities from repeats
p_gt_phoneme = np.array([all_results_meta[(plot_key, repeat)].p_gt_phoneme
                        for repeat in range(n_repeats)]).T
p_gt_lexical_evidence = np.array([all_results_meta[(plot_key, repeat)].p_gt_lexical_evidence
                              for repeat in range(n_repeats)]).T

sns.regplot(x=p_gt_phoneme.mean(1), y=p_gt_lexical_evidence.mean(1))

In [ ]:
np.corrcoef(p_gt_phoneme.mean(1), p_gt_lexical_evidence.mean(1))[0, 1]

## Plot B-population HGA

In [ ]:
plot_key = "EC260_dn_A5_B1"
plot_meta = all_results_meta[plot_key, 0]
plot_num_quantiles = 4

# sanity check: test trials are the same across repeats
test_trial_indices = np.array([all_results_meta[(plot_key, repeat)].test_trial_metadata.index
                                for repeat in range(n_repeats)]).T
for i in range(1, test_trial_indices.shape[1]):
    np.testing.assert_array_equal(
        test_trial_indices[:, i],
        test_trial_indices[:, 0],
    )

# merge estimated probabilities from repeats
p_gt_phoneme = np.array([all_results_meta[(plot_key, repeat)].p_gt_phoneme
                        for repeat in range(n_repeats)]).T
p_gt_phoneme_binned = pd.qcut(p_gt_phoneme.mean(1), plot_num_quantiles)
p_gt_lexical_evidence = np.array([all_results_meta[(plot_key, repeat)].p_gt_lexical_evidence
                              for repeat in range(n_repeats)]).T

plot_meta_df = plot_meta.test_trial_metadata.copy()
plot_meta_df["p_gt_phoneme"] = p_gt_phoneme.mean(1)
plot_meta_df["p_gt_lexical_evidence"] = p_gt_lexical_evidence.mean(1)
plot_meta_df["p_gt_phoneme_binned"] = p_gt_phoneme_binned
plot_meta_df["p_gt_phoneme_bin_center"] = plot_meta_df.p_gt_phoneme_binned.apply(
    lambda x: x.mid
).astype(float)

def get_textgrid_path(row):
    return Path(tg_dir) / (Path(row.wav_file).with_suffix(".TextGrid").name)
plot_meta_df["textgrid_path"] = plot_meta_df.apply(get_textgrid_path, axis=1)

In [ ]:
from matplotlib import transforms


g = sns.FacetGrid(
    pd.merge(plot_meta_df,
             electrode_df.loc[plot_meta.subject].loc[all_results_meta[plot_key, 0].population_B].reset_index(),
             how="cross"),
    row="electrode_idx",
    hue="p_gt_phoneme_bin_center", palette="viridis",
    col="lexical_evidence",
    aspect=3, height=2, sharey="row")

def plot_facet(data, color, **kwargs):
    electrode_idx = data.electrode_idx.iloc[0]
    epoch_idxs = data.index

    word_end = data.word_end.iloc[0]
    tg_path = data.textgrid_path.iloc[0]
    tg = textgrid.TextGrid.fromFile(str(tg_path))

    ax = plt.gca()
    ax.set_xlabel("Time since word onset (sec)")
    ax.set_ylabel("HGA")
    ax.set_title(f"Electrode {electrode_idx}, {word_end}")

    ax.axvspan(*plot_meta.population_B_window, color="gray", alpha=0.2)

    # plot epoched response at this electrode
    plot_epochs = plot_meta.epochs[epoch_idxs]
    plot_epoch_data = plot_epochs.copy().pick(electrode_idx).get_data().squeeze(1)
    assert plot_epoch_data.ndim == 2  # n_trials * n_times

    plot_times = plot_epochs.times
    plot_epoch_data_mean = plot_epoch_data.mean(0)
    plot_epoch_data_sem = plot_epoch_data.std(0) / np.sqrt(plot_epoch_data.shape[0])
    ax.plot(plot_times, plot_epoch_data_mean, color=color, alpha=0.5, **kwargs)
    ax.fill_between(plot_times, plot_epoch_data_mean - plot_epoch_data_sem,
                    plot_epoch_data_mean + plot_epoch_data_sem, color=color, alpha=0.2)
    
    # ylim = ax.get_ylim()
    # for i in range(len(plot_epoch_data)):
    #     ax.plot(plot_times, plot_epoch_data[i], color=color, alpha=0.1)
    # ax.set_ylim(ylim)
    
    intervals = [interval for interval in tg.tiers[0].intervals
                 if interval.mark is not None and interval.mark.strip()]
    for i, interval in enumerate(intervals):
        if interval.mark is None or not interval.mark.strip():
                continue
        ax.axvline(interval.minTime, linestyle="--", alpha=0.5, color="salmon")
        ax.text(interval.minTime, 0.025, interval.mark.strip(), rotation=90,
                ha="right", va="bottom",
                transform=transforms.blended_transform_factory(ax.transData, ax.transAxes))
        
        if i == len(intervals) - 1:
            # plot offset as well.
            ax.axvline(interval.maxTime, linestyle="--", alpha=0.5, color="blue")

    return ax

g.map_dataframe(plot_facet)
g.add_legend()

## Start a new searchlight

Rather than relying on B-populations with fixed time windows, we will use a more hypothesis-free method, simply running windowed t-tests looking for contrasts by extreme high vs low. p-gt-phoneme values.

In [ ]:
# searchlight_results = {}
# searchlight_window_size = 0.2
# searchlight_window_start = 0.5
# searchlight_window_end = 3.5
# searchlight_num_quantiles = 3

# searchlight_method = "spearmanr"  # ttest, spearmanr, or pearsonr

# for experiment in tqdm(all_results_df.index.get_level_values("name").unique()):
#     subject, phoneme_pair, population_A, population_B = experiment.split("_")
#     population_A = int(population_A[1:])
#     result_key = (subject, phoneme_pair, population_A)
#     if result_key in searchlight_results:
#         continue

#     prev_results = all_results_meta[experiment, 0]

#     searchlight_window_size_samp = int(searchlight_window_size * prev_results.epochs.info["sfreq"])
#     searchlight_window_start_samp = int(searchlight_window_start * prev_results.epochs.info["sfreq"])
#     searchlight_window_end_samp = int(searchlight_window_end * prev_results.epochs.info["sfreq"])

#     p_gt_phoneme = np.array([all_results_meta[experiment, repeat].p_gt_phoneme
#                              for repeat in range(n_repeats)]).T
#     p_gt_phoneme_mean = p_gt_phoneme.mean(1)
#     p_gt_phoneme_binned = pd.qcut(p_gt_phoneme_mean, searchlight_num_quantiles)

#     window_starts = np.arange(
#         searchlight_window_start_samp,
#         searchlight_window_end_samp - searchlight_window_size_samp,
#         searchlight_window_size_samp
#     )
#     window_ends = window_starts + searchlight_window_size_samp

#     if searchlight_method in ["spearmanr", "pearsonr"]:
#         eval_epochs = prev_results.epochs[prev_results.test_trial_metadata.index]

#         mask_left = eval_epochs.metadata.lexical_evidence == 0

#         eval_epochs_left = eval_epochs[mask_left]
#         eval_epochs_right = eval_epochs[~mask_left]

#         eval_epochs_left = eval_epochs_left.get_data()
#         eval_epochs_right = eval_epochs_right.get_data()

#         for window_start, window_end in zip(window_starts, window_ends):
#             # get the data for this window
#             window_data_left = eval_epochs_left[:, :, window_start:window_end]
#             window_data_right = eval_epochs_right[:, :, window_start:window_end]

#             # average across time
#             window_data_left_mean = window_data_left.mean(2)
#             window_data_right_mean = window_data_right.mean(2)

#             results = {}
#             for side, window_data_mean, p_gt_phoneme_mean_side in [("left", window_data_left_mean, p_gt_phoneme_mean[mask_left]),
#                                                                    ("right", window_data_right_mean, p_gt_phoneme_mean[~mask_left])]:
#                 if searchlight_method == "spearmanr":
#                     side_test = [spearmanr(window_data_mean[:, i], p_gt_phoneme_mean_side)
#                                  for i in range(window_data_mean.shape[1])]
#                     corr, pval = zip(*side_test)
#                 elif searchlight_method == "pearsonr":
#                     corr, pval = pearsonr(window_data_mean, p_gt_phoneme_mean_side[:, None], axis=0)
#                 results[side] = (corr, pval)

#             # save results
#             searchlight_results[*result_key, window_start] = pd.DataFrame({
#                 "corr_left": results["left"][0],
#                 "p_val_left": results["left"][1],
#                 "corr_right": results["right"][0],
#                 "p_val_right": results["right"][1],
#                 "electrode_idx": np.arange(len(results["left"][0])),
#                 "window_start_samp": window_start,
#                 "window_end_samp": window_end,
#                 "window_start": prev_results.epochs.times[window_start],
#                 "window_end": prev_results.epochs.times[window_end],
#                 "population_A": population_A,
#                 "phoneme_pair": phoneme_pair,
#                 "subject": subject,
#             })
#     elif searchlight_method == "ttest":
#         left_extreme, right_extreme = p_gt_phoneme_binned.categories[[0, -1]]

#         left_md = prev_results.test_trial_metadata[p_gt_phoneme_binned == left_extreme]
#         right_md = prev_results.test_trial_metadata[p_gt_phoneme_binned == right_extreme]

#         left_epochs = prev_results.epochs[left_md.index]
#         right_epochs = prev_results.epochs[right_md.index]

#         left_data = left_epochs.get_data()
#         right_data = right_epochs.get_data()

#         for window_start, window_end in zip(window_starts, window_ends):
#             # get the data for this window
#             left_window_data = left_data[:, :, window_start:window_end]
#             right_window_data = right_data[:, :, window_start:window_end]

#             # average across time
#             left_window_data_mean = left_window_data.mean(2)
#             right_window_data_mean = right_window_data.mean(2)

#             # t-test
#             t_stat, p_val = ttest_ind(left_window_data_mean, right_window_data_mean, axis=0)

#             # save results
#             searchlight_results[*result_key, window_start] = pd.DataFrame({
#                 "t_stat": t_stat,
#                 "p_val": p_val,
#                 "electrode_idx": np.arange(len(t_stat)),
#                 "window_start_samp": window_start,
#                 "window_end_samp": window_end,
#                 "population_A": population_A,
#                 "phoneme_pair": phoneme_pair,
#                 "subject": subject,
#             })

In [ ]:
searchlight_results = {}
searchlight_window_size = 0.2
searchlight_window_start = 0.5
searchlight_window_end = 3.5
searchlight_num_quantiles = 3

searchlight_method = "spearmanr"  # ttest, spearmanr, or pearsonr

# map from unique experiment key to a key on `all_results_meta` -- which may have some varying
# B population which we don't care about here.
searchlight_experiments = {}
for experiment in all_results_df.index.get_level_values("name").unique():
    subject, phoneme_pair, population_A, population_B = experiment.split("_")
    population_A = int(population_A[1:])
    result_key = (subject, phoneme_pair, population_A)
    searchlight_experiments[result_key] = experiment

# pre-compute p(gt phoneme) for all experiments
searchlight_decoder_outputs = {}
for experiment, orig_name in searchlight_experiments.items():
    subject, phoneme_pair, population_A = experiment
    
    p_gt_phoneme = np.array([all_results_meta[(orig_name, repeat)].p_gt_phoneme
                             for repeat in range(n_repeats)]).T
    p_gt_phoneme_mean = p_gt_phoneme.mean(1)

    searchlight_decoder_outputs[experiment] = {
        "p_gt_phoneme_mean": p_gt_phoneme_mean,
    }

searchlight_electrode_activations = {}
for experiment, orig_name in tqdm(searchlight_experiments.items()):
    subject, phoneme_pair, population_A = experiment
    result_key = (subject, phoneme_pair, population_A)
    prev_results = all_results_meta[orig_name, 0]

    searchlight_window_size_samp = int(searchlight_window_size * prev_results.epochs.info["sfreq"])
    searchlight_window_start_samp = int(searchlight_window_start * prev_results.epochs.info["sfreq"])
    searchlight_window_end_samp = int(searchlight_window_end * prev_results.epochs.info["sfreq"])

    p_gt_phoneme_mean = searchlight_decoder_outputs[experiment]["p_gt_phoneme_mean"]
    p_gt_phoneme_binned = pd.qcut(p_gt_phoneme_mean, searchlight_num_quantiles)

    window_starts = np.arange(
        searchlight_window_start_samp,
        searchlight_window_end_samp - searchlight_window_size_samp,
        searchlight_window_size_samp
    )
    window_ends = window_starts + searchlight_window_size_samp

    if searchlight_method in ["spearmanr", "pearsonr"]:
        eval_epochs = prev_results.epochs[prev_results.test_trial_metadata.index]

        mask_left = eval_epochs.metadata.lexical_evidence == 0

        eval_epochs_left = eval_epochs[mask_left]
        eval_epochs_right = eval_epochs[~mask_left]

        eval_epochs_left = eval_epochs_left.get_data()
        eval_epochs_right = eval_epochs_right.get_data()

        for window_start, window_end in zip(window_starts, window_ends):
            # get the data for this window
            window_data_left = eval_epochs_left[:, :, window_start:window_end]
            window_data_right = eval_epochs_right[:, :, window_start:window_end]

            # average across time
            window_data_left_mean = window_data_left.mean(2)
            window_data_right_mean = window_data_right.mean(2)

            searchlight_electrode_activations[*experiment, window_start, window_end] = {
                "metadata": prev_results.test_trial_metadata,
                "mask_left": mask_left,
                "window_data_left_mean": window_data_left_mean,
                "window_data_right_mean": window_data_right_mean,
            }

            results = {}
            for side, window_data_mean, p_gt_phoneme_mean_side in [("left", window_data_left_mean, p_gt_phoneme_mean[mask_left]),
                                                                   ("right", window_data_right_mean, p_gt_phoneme_mean[~mask_left])]:
                if searchlight_method == "spearmanr":
                    side_test = [spearmanr(window_data_mean[:, i], p_gt_phoneme_mean_side)
                                 for i in range(window_data_mean.shape[1])]
                    corr, pval = zip(*side_test)
                elif searchlight_method == "pearsonr":
                    corr, pval = pearsonr(window_data_mean, p_gt_phoneme_mean_side[:, None], axis=0)
                results[side] = (corr, pval)

            # save results
            searchlight_results[*result_key, window_start] = pd.DataFrame({
                "corr_left": results["left"][0],
                "p_val_left": results["left"][1],
                "corr_right": results["right"][0],
                "p_val_right": results["right"][1],
                "electrode_idx": np.arange(len(results["left"][0])),
                "window_start_samp": window_start,
                "window_end_samp": window_end,
                "window_start": prev_results.epochs.times[window_start],
                "window_end": prev_results.epochs.times[window_end],
                "population_A": population_A,
                "phoneme_pair": phoneme_pair,
                "subject": subject,
            })
    elif searchlight_method == "ttest":
        raise NotImplementedError("T-test searchlight not implemented in this refactor.")
        left_extreme, right_extreme = p_gt_phoneme_binned.categories[[0, -1]]

        left_md = prev_results.test_trial_metadata[p_gt_phoneme_binned == left_extreme]
        right_md = prev_results.test_trial_metadata[p_gt_phoneme_binned == right_extreme]

        left_epochs = prev_results.epochs[left_md.index]
        right_epochs = prev_results.epochs[right_md.index]

        left_data = left_epochs.get_data()
        right_data = right_epochs.get_data()

        for window_start, window_end in zip(window_starts, window_ends):
            # get the data for this window
            left_window_data = left_data[:, :, window_start:window_end]
            right_window_data = right_data[:, :, window_start:window_end]

            # average across time
            left_window_data_mean = left_window_data.mean(2)
            right_window_data_mean = right_window_data.mean(2)

            # t-test
            t_stat, p_val = ttest_ind(left_window_data_mean, right_window_data_mean, axis=0)

            # save results
            searchlight_results[*result_key, window_start] = pd.DataFrame({
                "t_stat": t_stat,
                "p_val": p_val,
                "electrode_idx": np.arange(len(t_stat)),
                "window_start_samp": window_start,
                "window_end_samp": window_end,
                "population_A": population_A,
                "phoneme_pair": phoneme_pair,
                "subject": subject,
            })

In [ ]:
searchlight_results_df = pd.concat(searchlight_results.values())
searchlight_results_df["sig_left"] = searchlight_results_df.p_val_left < 0.001
searchlight_results_df["sig_right"] = searchlight_results_df.p_val_right < 0.001

In [ ]:
# Only retain results for which we have electrode metadata
searchlight_results_df = electrode_df.merge(searchlight_results_df, left_index=True, right_on=["subject", "electrode_idx"], how="inner")
searchlight_results_df["p_val_min"] = searchlight_results_df[["p_val_left", "p_val_right"]].min(axis=1)
searchlight_results_df = searchlight_results_df.sort_values("p_val_min").head(200)

In [ ]:
searchlight_results_df

In [ ]:
# plot_searchlight_idx = 2
# plot_row = searchlight_results_df.sort_values("p_val_min").iloc[plot_searchlight_idx]
# print(plot_row)
# plot_searchlight_response(plot_row.subject, plot_row.population_A, plot_row.phoneme_pair,
#                           [plot_row.electrode_idx], (plot_row.window_start_samp, plot_row.window_end_samp))
# #"EC260", 5, "dn", [18], (210, 230))

In [ ]:
def plot_searchlight_response(subject, population_A, phoneme_pair, population_B, population_B_window):
    plot_key = next(key for key in all_results_df.index.get_level_values("name") if key.startswith(f"{subject}_{phoneme_pair}_A{population_A}_"))
    plot_num_quantiles = 4
    plot_meta = all_results_meta[plot_key, 0]

    # sanity check: test trials are the same across repeats
    test_trial_indices = np.array([all_results_meta[(plot_key, repeat)].test_trial_metadata.index
                                    for repeat in range(n_repeats)]).T
    for i in range(1, test_trial_indices.shape[1]):
        np.testing.assert_array_equal(
            test_trial_indices[:, i],
            test_trial_indices[:, 0],
        )

    # merge estimated probabilities from repeats
    p_gt_phoneme = np.array([all_results_meta[(plot_key, repeat)].p_gt_phoneme
                            for repeat in range(n_repeats)]).T
    p_gt_phoneme_binned = pd.qcut(p_gt_phoneme.mean(1), plot_num_quantiles)
    p_gt_lexical_evidence = np.array([all_results_meta[(plot_key, repeat)].p_gt_lexical_evidence
                                for repeat in range(n_repeats)]).T

    plot_meta_df = plot_meta.test_trial_metadata.copy()
    plot_meta_df["p_gt_phoneme"] = p_gt_phoneme.mean(1)
    plot_meta_df["p_gt_lexical_evidence"] = p_gt_lexical_evidence.mean(1)
    plot_meta_df["p_gt_phoneme_binned"] = p_gt_phoneme_binned
    plot_meta_df["p_gt_phoneme_bin_center"] = plot_meta_df.p_gt_phoneme_binned.apply(
        lambda x: x.mid
    ).astype(float).round(3)

    def get_textgrid_path(row):
        return Path(tg_dir) / (Path(row.wav_file).with_suffix(".TextGrid").name)
    plot_meta_df["textgrid_path"] = plot_meta_df.apply(get_textgrid_path, axis=1)
    plot_meta_df = plot_meta_df.rename_axis("epoch_idx").reset_index()

    # cross by electrodes
    plot_meta_df = pd.merge(plot_meta_df,
                 electrode_df.loc[plot_meta.subject].loc[population_B].reset_index(),
                 how="cross")

    ####

    g_scatter = sns.FacetGrid(
        plot_meta_df,
        row="electrode_idx",
        col="lexical_evidence",
        aspect=1, height=3, sharey="row")
    
    def plot_facet_scatter(data, color, **kwargs):
        electrode_idx = data.electrode_idx.iloc[0]
        epoch_idxs = data.epoch_idx

        word_end = data.word_end.iloc[0]
        tg_path = data.textgrid_path.iloc[0]

        ax = plt.gca()
        ax.set_xlabel("Estimated phoneme probability")
        ax.set_ylabel("HGA")
        ax.set_title(f"{subject} {electrode_idx + 1}, {word_end}")

        # plot epoched response at this electrode
        plot_epochs = plot_meta.epochs[epoch_idxs]
        plot_epoch_data = plot_epochs.copy().pick(electrode_idx).get_data()
        
        window_start_samp, window_end_samp = population_B_window
        plot_epoch_data = plot_epoch_data.squeeze(1)[:, window_start_samp:window_end_samp]
        plot_epoch_data = plot_epoch_data.mean(1)

        assert plot_epoch_data.ndim == 1  # n_trials
        
        sns.regplot(
            x=data.p_gt_phoneme,
            y=plot_epoch_data,
            ax=ax,
        )

        corr = np.corrcoef(data.p_gt_phoneme, plot_epoch_data)[0, 1]
        ax.text(0.05, 0.95, f"r = {corr:.2f}",
                transform=ax.transAxes, ha="left", va="top",
                bbox=dict(facecolor="white", alpha=0.5, edgecolor="none"))

        return ax

    g_scatter.map_dataframe(plot_facet_scatter)

    ####

    g = sns.FacetGrid(
        plot_meta_df,
        row="electrode_idx",
        hue="p_gt_phoneme_bin_center", palette="plasma",
        col="lexical_evidence",
        aspect=3, height=2, sharey="row")

    def plot_facet(data, color, **kwargs):
        electrode_idx = data.electrode_idx.iloc[0]
        epoch_idxs = data.epoch_idx

        word_end = data.word_end.iloc[0]
        tg_path = data.textgrid_path.iloc[0]
        tg = textgrid.TextGrid.fromFile(str(tg_path))

        ax = plt.gca()
        ax.set_xlabel("Time since word onset (sec)")
        ax.set_ylabel("HGA")
        ax.set_title(f"{subject} {electrode_idx + 1}, {word_end}")

        # plot epoched response at this electrode
        plot_epochs = plot_meta.epochs[epoch_idxs]
        plot_epoch_data = plot_epochs.copy().pick(electrode_idx).get_data().squeeze(1)
        assert plot_epoch_data.ndim == 2  # n_trials * n_times

        ax.axvspan(*plot_epochs.times[list(population_B_window)], color="gray", alpha=0.2)

        plot_times = plot_epochs.times
        plot_epoch_data_mean = plot_epoch_data.mean(0)
        plot_epoch_data_sem = plot_epoch_data.std(0) / np.sqrt(plot_epoch_data.shape[0])
        ax.plot(plot_times, plot_epoch_data_mean, color=color, alpha=0.5, **kwargs)
        ax.fill_between(plot_times, plot_epoch_data_mean - plot_epoch_data_sem,
                        plot_epoch_data_mean + plot_epoch_data_sem, color=color, alpha=0.2)
        
        ax.set_xlim(plot_epochs.times[0], plot_epochs.times[-1])
        
        # ylim = ax.get_ylim()
        # for i in range(len(plot_epoch_data)):
        #     ax.plot(plot_times, plot_epoch_data[i], color=color, alpha=0.1)
        # ax.set_ylim(ylim)
        
        intervals = [interval for interval in tg.tiers[0].intervals
                    if interval.mark is not None and interval.mark.strip()]
        for i, interval in enumerate(intervals):
            if interval.mark is None or not interval.mark.strip():
                    continue
            ax.axvline(interval.minTime, linestyle="--", alpha=0.5, color="salmon")
            ax.text(interval.minTime, 0.025, interval.mark.strip(), rotation=90,
                    ha="right", va="bottom",
                    transform=transforms.blended_transform_factory(ax.transData, ax.transAxes))
            
            if i == len(intervals) - 1:
                # plot offset as well.
                ax.axvline(interval.maxTime, linestyle="--", alpha=0.5, color="blue")

        return ax

    g.map_dataframe(plot_facet)

    # Add vertical lines for behavioral RT distributions
    for i, row in enumerate(g.row_names):
        for j, col in enumerate(g.col_names):
            ax = g.axes[i, j]
            subplot_df = plot_meta_df[(plot_meta_df.electrode_idx == int(row)) & (plot_meta_df.lexical_evidence == col)]

            value = subplot_df["slider.rt"]
            ax.axvline(value.median(), color="forestgreen", linestyle="--", alpha=0.5)
            # plot IQR
            ax.axvspan(
                value.quantile(0.25),
                value.quantile(0.75),
                color="forestgreen", alpha=0.2
            )

    g.add_legend()

    return g, g_scatter

In [ ]:
# DEV
plot_row = searchlight_results_df.sort_values("p_val_min").iloc[0]
plot_searchlight_response(
    plot_row.subject, plot_row.population_A, plot_row.phoneme_pair,
    [plot_row.electrode_idx], (plot_row.window_start_samp, plot_row.window_end_samp)
)
None

In [ ]:
def evaluate_counterfactual_correlation(A_source, B_source, B_electrode_idx, B_window_start, B_window_end, left=True):
    A_subject, A_phoneme_pair, A_population_A = A_source
    B_subject, B_phoneme_pair, B_population_A = B_source

    assert A_phoneme_pair == B_phoneme_pair, "Phoneme pairs must match for counterfactual evaluation."

    A_activations = searchlight_electrode_activations[A_subject, A_phoneme_pair, A_population_A, B_window_start, B_window_end]
    B_activations = searchlight_electrode_activations[B_subject, B_phoneme_pair, B_population_A, B_window_start, B_window_end]

    A_metadata = A_activations["metadata"].copy()
    B_metadata = B_activations["metadata"]
    A_mask_left = A_activations["mask_left"]
    B_mask_left = B_activations["mask_left"]

    if left:
        A_metadata = A_metadata[A_mask_left]
        B_metadata = B_metadata[B_mask_left]

        A_p_gt_phoneme = searchlight_decoder_outputs[A_subject, A_phoneme_pair, A_population_A]["p_gt_phoneme_mean"][A_mask_left]
        B_response = B_activations["window_data_left_mean"][:, B_electrode_idx]
    else:
        A_metadata = A_metadata[~A_mask_left]
        B_metadata = B_metadata[~B_mask_left]

        A_p_gt_phoneme = searchlight_decoder_outputs[A_subject, A_phoneme_pair, A_population_A]["p_gt_phoneme_mean"][~A_mask_left]
        B_response = B_activations["window_data_right_mean"][:, B_electrode_idx]

    if len(A_metadata) != len(B_metadata):
        # resample A to have the same number of trials as B
        B_grouped = B_metadata.groupby(["resampled", "lexical_evidence"])
        draw_A_idxs = A_metadata.groupby(["resampled", "lexical_evidence"], as_index=False).apply(
                lambda xs: xs.iloc[np.random.choice(len(xs),
                                                    size=len(B_grouped.groups[xs.name]),
                                                    replace=len(B_grouped.groups[xs.name]) > len(xs))]) \
            .reset_index(level=0, drop=True).sort_values(["resampled", "lexical_evidence"]).index
        A_metadata["local_idx"] = np.arange(len(A_metadata))
        A_metadata = A_metadata.loc[draw_A_idxs]
        A_p_gt_phoneme = A_p_gt_phoneme[A_metadata.local_idx]
    else:
        # resort A
        A_metadata = A_metadata.reset_index().sort_values(["resampled", "lexical_evidence"])
        A_p_gt_phoneme = A_p_gt_phoneme[A_metadata.index]
    
    # align
    B_resorted = B_metadata.reset_index().sort_values(["resampled", "lexical_evidence"]).index
    B_response = B_response[B_resorted]

    return spearmanr(A_p_gt_phoneme, B_response)

In [ ]:
# A_subject, A_phoneme_pair, A_population_A = "EC260", "dn", 5
# B_subject, B_phoneme_pair, B_population_A = "EC287", "dn", 5

# A_activations = searchlight_electrode_activations[A_subject, A_phoneme_pair, A_population_A, 210, 230]
# B_activations = searchlight_electrode_activations[B_subject, B_phoneme_pair, B_population_A, 210, 230]

# A_metadata = A_activations["metadata"]
# B_metadata = B_activations["metadata"]

# A_mask_left = A_activations["mask_left"]
# B_mask_left = B_activations["mask_left"]

# A_metadata = A_metadata[A_mask_left]
# B_metadata = B_metadata[B_mask_left]

# A_metadata["local_idx"] = np.arange(len(A_metadata))

# B_grouped = B_metadata.groupby(["resampled", "lexical_evidence"])
# draw_A_idxs = A_metadata.groupby(["resampled", "lexical_evidence"], as_index=False).apply(lambda xs: xs.iloc[np.random.choice(len(xs), size=len(B_grouped.groups[xs.name]), replace=True)]) \
#     .reset_index(level=0, drop=True).sort_values(["resampled", "lexical_evidence"]).index
# A_metadata = A_metadata.loc[draw_A_idxs]

# A_p_gt_phoneme = searchlight_decoder_outputs[A_subject, A_phoneme_pair, A_population_A]["p_gt_phoneme_mean"][A_mask_left]
# A_p_gt_phoneme = A_p_gt_phoneme[A_metadata.local_idx]

In [ ]:
def counterfactual_baseline(subject, phoneme_pair, population_A, electrode_idx, window_start, window_end, left=True,
                            sanity_check=False):
    if sanity_check:
        # DEV sanity check: only use the same subject and phoneme pair
        control_alternative_keys = [(subject_alt, phoneme_pair_alt, population_A_alt) for subject_alt, phoneme_pair_alt, population_A_alt in searchlight_decoder_outputs.keys()
                                    if subject_alt == subject and phoneme_pair_alt == phoneme_pair]
    else:
        control_alternative_keys = [(subject_alt, phoneme_pair_alt, population_A_alt) for subject_alt, phoneme_pair_alt, population_A_alt in searchlight_decoder_outputs.keys()
                                    if subject_alt != subject and phoneme_pair_alt == phoneme_pair]

    ret = []
    for key in control_alternative_keys:
        try:
            val = evaluate_counterfactual_correlation(
                (subject, phoneme_pair, population_A),
                key,
                electrode_idx,
                window_start,
                window_end,
                left=left
            )
            ret.append((key, val))
        except Exception as e:
            print(f"Error evaluating counterfactual for {key}: {e}")
            continue
    return ret

In [ ]:
control_row = searchlight_results_df.sort_values("p_val_min").iloc[4]
counterfactual_baseline(control_row.subject, control_row.phoneme_pair, control_row.population_A,
                        control_row.electrode_idx, control_row.window_start_samp, control_row.window_end_samp, left=control_row.sig_left)

In [ ]:
evaluate_counterfactual_correlation(control_alternative, control_key, control_row.electrode_idx,
                                     control_row.window_start_samp, control_row.window_end_samp, left=False)

In [ ]:
# but we need to make sure the trials are ordered the same way
prev_results.test_trial_metadata.resampled

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages("searchlight_results.pdf") as pdf:
    for _, row in tqdm(searchlight_results_df.sort_values("p_val_min").iterrows(), total=len(searchlight_results_df)):
        g_time_series, g_scatter = plot_searchlight_response(
            row.subject, row.population_A, row.phoneme_pair,
            [row.electrode_idx], (row.window_start_samp, row.window_end_samp)
        )

        pdf.savefig(g_time_series.fig)
        pdf.savefig(g_scatter.fig)
        plt.close(g_time_series.fig)
        plt.close(g_scatter.fig)

In [ ]:
searchlight_results_df.to_csv("causal4_searchlight_results.csv")

In [ ]:
# Load results with manual annotations
searchlight_manual = pd.read_csv("causal4_searchlight_results_manual.csv", index_col=0)

In [ ]:
if not (searchlight_manual.index == searchlight_results_df.index).all():
    raise ValueError("Manual annotations do not match the current searchlight results. "
                     "Probably need to repeat manual annotations.")

In [ ]:
plot_subject = "EC260"
plot_electrode_idx = 18
plot_phoneme_pair = "dn"
plot_population_A = 5
plot_key = next(iter(key for key in all_results_df.index.get_level_values("name")
                     if key.startswith(f"{plot_subject}_{plot_phoneme_pair}_A{plot_population_A}_")))

plot_results = all_results_meta[plot_key, 0]
plot_ep = plot_results.epochs[plot_results.test_trial_metadata.index]
p_gt_phoneme = np.array([all_results_meta[(plot_key, repeat)].p_gt_phoneme
                            for repeat in range(n_repeats)]).T
p_gt_phoneme = p_gt_phoneme.mean(1)

rts = plot_ep.metadata["slider.rt"].combine(plot_ep.tmax, min)
plot_ep.plot_image(
    picks=[plot_electrode_idx],
    # overlay_times=rts,
    order=p_gt_phoneme.argsort())

In [ ]:
prev_results.epochs.plot_image(picks=[])

In [ ]:
searchlight_manual